In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
import pydicom
from PIL import Image

# Show attached Kaggle datasets
INPUT_DIR = Path("/kaggle/input")

print("Attached inputs:")
for folder in INPUT_DIR.iterdir():
    print("-", folder.name)

# Find saved CNN weights
print("\nSaved model weights:")
for path in INPUT_DIR.rglob("v4_selected.weights.h5"):
    print(path)

# Find competition sample submission
print("\nSample submission:")
for path in INPUT_DIR.rglob("sample_submission.csv"):
    print(path)

Attached inputs:
- notebooks
- competitions

Saved model weights:
/kaggle/input/notebooks/ericzhang15/ez-mri-training-evaluation/v4_selected.weights.h5

Sample submission:
/kaggle/input/competitions/rsna-knee-abnormality-detection/sample_submission.csv


In [2]:
# ===================================
# Rebuild V4 CNN and load saved weights
# ===================================

IMG_SIZE = (160, 160)

LABEL_COLS = [
    'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus',
    'Medial OA', 'Lateral OA', 'PF OA', 'Effusion',
    'Synovitis', "Baker's", 'Contusion', 'Fracture'
]

def build_model():
    inputs = tf.keras.Input(shape=(*IMG_SIZE, 3), name='mri_slices')

    x = tf.keras.layers.Rescaling(1 / 127.5, offset=-1)(inputs)

    backbone = tf.keras.applications.MobileNetV2(
        include_top=False,
        weights=None,
        input_shape=(*IMG_SIZE, 3)
    )

    backbone.trainable = False
    x = backbone(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dense(256, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.35)(x)

    outputs = tf.keras.layers.Dense(
        len(LABEL_COLS), activation='sigmoid'
    )(x)

    return tf.keras.Model(inputs, outputs)


# Recreate the same architecture
model = build_model()

# Load the weights trained in Notebook 1
WEIGHTS_PATH = Path(
    '/kaggle/input/notebooks/ericzhang15/'
    'ez-mri-training-evaluation/v4_selected.weights.h5'
)

model.load_weights(str(WEIGHTS_PATH))

print("Trained V4 weights loaded successfully!")
print("Model output shape:", model.output_shape)

I0000 00:00:1792616531.027515      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1792616531.031067      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Trained V4 weights loaded successfully!
Model output shape: (None, 12)


Step 3 — Load and preprocess test MRIs

In [3]:
import hashlib

DATA_DIR = Path('/kaggle/input/competitions/rsna-knee-abnormality-detection')
CACHE_DIR = Path('/kaggle/working/mri_cache_submission')
CACHE_DIR.mkdir(parents=True, exist_ok=True)

NUM_SLICES = 3

test_df = pd.read_csv(DATA_DIR / 'test.csv')
test_series_df = pd.read_csv(DATA_DIR / 'test_series.csv')
sample_submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')


# Choose a sagittal, preferably fluid-sensitive, MRI series
def select_series(study_uid, series_df):
    matches = series_df.loc[
        series_df['StudyInstanceUID'].astype(str) == str(study_uid)
    ]

    if matches.empty:
        raise FileNotFoundError(f'No MRI series for {study_uid}')

    sagittal = matches['Anatomical_Plane'].astype(str).str.lower().eq('sagittal')
    fluid = pd.to_numeric(
        matches['Fluid_Sensitive'], errors='coerce'
    ).eq(1)

    preferred = matches.loc[sagittal & fluid]

    if preferred.empty:
        preferred = matches.loc[sagittal]

    if preferred.empty:
        preferred = matches

    return str(preferred.iloc[0]['SeriesInstanceUID'])


# Normalize MRI pixel intensities to 0–255
def normalize_image(pixels):
    pixels = np.asarray(pixels, dtype=np.float32)
    lo, hi = np.percentile(pixels, [1, 99])

    if hi <= lo:
        return np.zeros(pixels.shape, dtype=np.uint8)

    return (
        (np.clip(pixels, lo, hi) - lo) / (hi - lo) * 255
    ).astype(np.uint8)


# Convert one MRI study into a (160, 160, 3) CNN input
def load_study_image(study_uid, series_df, split='test'):
    cache_key = hashlib.sha256(
        f'v4|{split}|{study_uid}|{IMG_SIZE}|{NUM_SLICES}'.encode()
    ).hexdigest()[:24]

    cache_path = CACHE_DIR / f'{cache_key}.npy'

    if cache_path.exists():
        return np.load(cache_path)

    series_uid = select_series(study_uid, series_df)
    folder = DATA_DIR / f'{split}_series' / str(study_uid) / series_uid

    if not folder.is_dir():
        raise FileNotFoundError(f'Cannot find {folder}')

    ordered = []

    for path in folder.iterdir():
        if not path.is_file():
            continue

        try:
            hdr = pydicom.dcmread(
                str(path),
                stop_before_pixels=True,
                specific_tags=['InstanceNumber']
            )
            ordered.append((
                int(getattr(hdr, 'InstanceNumber', 0)),
                path.name,
                path
            ))
        except Exception:
            continue

    if not ordered:
        raise ValueError(f'No readable DICOM files in {folder}')

    ordered.sort(key=lambda item: (item[0], item[1]))

    indices = np.round(
        np.linspace(0.2, 0.8, NUM_SLICES) * (len(ordered) - 1)
    ).astype(int)

    channels = []

    for index in indices:
        dicom = pydicom.dcmread(str(ordered[index][2]))
        pixels = np.asarray(dicom.pixel_array, dtype=np.float32)

        if pixels.ndim != 2:
            raise ValueError(f'Expected 2D MRI slice, got {pixels.shape}')

        pixels = (
            pixels * float(getattr(dicom, 'RescaleSlope', 1.0))
            + float(getattr(dicom, 'RescaleIntercept', 0.0))
        )

        image = Image.fromarray(normalize_image(pixels))
        image = image.resize(IMG_SIZE[::-1], Image.Resampling.BILINEAR)

        channels.append(np.asarray(image, dtype=np.uint8))

    result = np.stack(channels, axis=-1)
    np.save(cache_path, result)

    return result


# Test preprocessing on one MRI study
example_uid = str(test_df.iloc[0]['StudyInstanceUID'])
example_image = load_study_image(example_uid, test_series_df)

print('Test studies:', len(test_df))
print('Example image shape:', example_image.shape)

Test studies: 3
Example image shape: (160, 160, 3)


In [4]:
# ===================================
# Generate Kaggle submission
# ===================================

from tqdm.auto import tqdm

BATCH_SIZE = 16
predictions = []
study_ids = []

# Process test MRIs in batches to limit memory usage
for start in tqdm(range(0, len(test_df), BATCH_SIZE),
                  desc="Predicting test MRIs"):

    batch = test_df.iloc[start:start + BATCH_SIZE]
    images = []

    for uid in batch["StudyInstanceUID"].astype(str):
        images.append(load_study_image(uid, test_series_df))
        study_ids.append(uid)

    X_batch = np.stack(images)
    batch_pred = model.predict(X_batch, verbose=0)
    predictions.append(batch_pred)

# Create a table containing the 12 prediction probabilities
prob = np.concatenate(predictions, axis=0)

submission_predictions = pd.DataFrame(prob, columns=LABEL_COLS)
submission_predictions.insert(0, "StudyInstanceUID", study_ids)

# Match the competition's required row and column order
submission = sample_submission[["StudyInstanceUID"]].merge(
    submission_predictions,
    on="StudyInstanceUID",
    how="left",
    validate="one_to_one"
)

# Check submission format and prediction values
assert submission.shape == sample_submission.shape
assert submission.columns.tolist() == sample_submission.columns.tolist()
assert submission[LABEL_COLS].notna().all().all()
assert np.isfinite(submission[LABEL_COLS].to_numpy()).all()
assert submission[LABEL_COLS].ge(0).all().all()
assert submission[LABEL_COLS].le(1).all().all()

# Save CSV
submission.to_csv("/kaggle/working/submission.csv", index=False)

print("Submission saved successfully!")
print("Submission shape:", submission.shape)
display(submission.head())

Predicting test MRIs:   0%|          | 0/1 [00:00<?, ?it/s]

2026-10-21 21:02:25.813020: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-10-21 21:02:25.956971: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-10-21 21:02:26.093230: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
I0000 00:00:1792616547.538436      71 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


Submission saved successfully!
Submission shape: (3, 13)


,StudyInstanceUID,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture
0,1.2.826.0.1.3680043.8.498.10047035057544427318...,0.067144,0.041797,0.448341,0.224265,0.467747,0.229616,0.636272,0.872313,0.687607,0.667883,0.282057,0.046757
1,1.2.826.0.1.3680043.8.498.10062861783145312629...,0.113087,0.094466,0.634710,0.229760,0.491654,0.462365,0.718279,0.440531,0.733711,0.279456,0.191158,0.115406
2,1.2.826.0.1.3680043.8.498.10067514707072572280...,0.482903,0.220104,0.632443,0.393875,0.485072,0.328942,0.608284,0.815789,0.769482,0.618206,0.542271,0.117691
